In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv("../data/raw/employee_data.csv")

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,34,No,Travel_Frequently,702,Research & Development,16,4,Life Sciences,1,838,...,3,80,0,6,3,3,5,2,1,3
1,38,No,Travel_Rarely,833,Research & Development,18,3,Medical,1,1766,...,3,80,1,15,2,3,1,0,1,0
2,51,No,Travel_Rarely,833,Research & Development,1,3,Life Sciences,1,353,...,2,80,0,1,0,2,1,0,0,0
3,60,No,Travel_Rarely,1179,Sales,16,4,Marketing,1,732,...,4,80,0,10,1,3,2,2,2,2
4,23,No,Travel_Rarely,571,Research & Development,12,2,Other,1,1982,...,3,80,0,5,6,4,5,2,1,4


In [3]:
X = df.drop(["Attrition", "EmployeeCount", "StandardHours", "Over18"], axis=1)
y = df["Attrition"]

In [4]:
# Separação entre variáveis independentes (X) e variável alvo (y).
# X contém as características dos colaboradores.
# y contém a variável Attrition, que indica se o colaborador abandonou ou não a empresa.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
# Divisão do dataset em treino e teste.
# 80% dos dados são usados para treinar o modelo e 20% para avaliar o desempenho.
# O parâmetro stratify=y mantém a mesma proporção de classes Attrition nos dois conjuntos.

In [7]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Numéricas:", list(numeric_features))
print("Categóricas:", list(categorical_features))

Numéricas: ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EmployeeNumber', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']
Categóricas: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


In [8]:
# Identificação automática das variáveis numéricas e categóricas.
# Esta separação é importante porque variáveis numéricas e categóricas
# precisam de transformações diferentes antes de serem usadas pelo modelo.

In [9]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [10]:
# Criação do pré-processamento dos dados.
#
# As variáveis numéricas são tratadas com SimpleImputer e StandardScaler.
# O SimpleImputer substitui possíveis valores em falta pela mediana.
# O StandardScaler normaliza os valores numéricos.
#
# As variáveis categóricas são tratadas com SimpleImputer e OneHotEncoder.
# O SimpleImputer substitui valores em falta pela categoria mais frequente.
# O OneHotEncoder transforma categorias de texto em valores numéricos.
#
# O ColumnTransformer junta estes dois processos e aplica cada tratamento
# ao tipo correto de variável.

In [11]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

logistic_model.fit(X_train, y_train)

logistic_y_pred = logistic_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, logistic_y_pred))
print("Logistic Regression F1 Score:", f1_score(y_test, logistic_y_pred, pos_label="Yes"))

print(confusion_matrix(y_test, logistic_y_pred))
print(classification_report(y_test, logistic_y_pred))

Logistic Regression Accuracy: 0.868
Logistic Regression F1 Score: 0.5217391304347826
[[199  11]
 [ 22  18]]
              precision    recall  f1-score   support

          No       0.90      0.95      0.92       210
         Yes       0.62      0.45      0.52        40

    accuracy                           0.87       250
   macro avg       0.76      0.70      0.72       250
weighted avg       0.86      0.87      0.86       250



In [12]:
# Treino e avaliação da Logistic Regression normal.
#
# Este modelo foi usado como baseline, ou seja, como primeiro modelo
# de comparação.
#
# A pipeline aplica automaticamente o pré-processamento aos dados
# antes de treinar a Logistic Regression.

In [13]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

rf_model.fit(X_train, y_train)

rf_y_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_y_pred))
print("Random Forest F1 Score:", f1_score(y_test, rf_y_pred, pos_label="Yes"))

print(confusion_matrix(y_test, rf_y_pred))
print(classification_report(y_test, rf_y_pred))

Random Forest Accuracy: 0.86
Random Forest F1 Score: 0.2857142857142857
[[208   2]
 [ 33   7]]
              precision    recall  f1-score   support

          No       0.86      0.99      0.92       210
         Yes       0.78      0.17      0.29        40

    accuracy                           0.86       250
   macro avg       0.82      0.58      0.60       250
weighted avg       0.85      0.86      0.82       250



In [14]:
# Treino e avaliação do modelo Random Forest.
#
# A Random Forest combina várias árvores de decisão para produzir
# previsões mais robustas.
#
# O parâmetro class_weight="balanced" foi utilizado para tentar reduzir
# o impacto do desequilíbrio entre as classes "No" e "Yes".

In [15]:
balanced_logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ]
)

balanced_logistic_model.fit(X_train, y_train)

balanced_y_pred = balanced_logistic_model.predict(X_test)

print("Balanced Logistic Regression Accuracy:", accuracy_score(y_test, balanced_y_pred))
print("Balanced Logistic Regression F1 Score:", f1_score(y_test, balanced_y_pred, pos_label="Yes"))

print(confusion_matrix(y_test, balanced_y_pred))
print(classification_report(y_test, balanced_y_pred))

Balanced Logistic Regression Accuracy: 0.732
Balanced Logistic Regression F1 Score: 0.45528455284552843
[[155  55]
 [ 12  28]]
              precision    recall  f1-score   support

          No       0.93      0.74      0.82       210
         Yes       0.34      0.70      0.46        40

    accuracy                           0.73       250
   macro avg       0.63      0.72      0.64       250
weighted avg       0.83      0.73      0.76       250



In [16]:
# Treino e avaliação da Logistic Regression com balanceamento de classes.
#
# O parâmetro class_weight="balanced" dá maior importância à classe
# minoritária, neste caso Attrition = "Yes".
#
# Esta abordagem foi testada porque o dataset possui mais colaboradores
# que permaneceram na empresa do que colaboradores que abandonaram.

In [17]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Balanced Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(y_test, logistic_y_pred),
        accuracy_score(y_test, rf_y_pred),
        accuracy_score(y_test, balanced_y_pred)
    ],
    "F1 Score Yes": [
        f1_score(y_test, logistic_y_pred, pos_label="Yes"),
        f1_score(y_test, rf_y_pred, pos_label="Yes"),
        f1_score(y_test, balanced_y_pred, pos_label="Yes")
    ]
})

results

,Model,Accuracy,F1 Score Yes
0,Logistic Regression,0.868,0.521739
1,Random Forest,0.860,0.285714
2,Balanced Logistic Regression,0.732,0.455285


In [18]:
# Comparação dos modelos testados.
# Foram comparadas as métricas Accuracy e F1 Score para a classe "Yes",
# que representa os colaboradores que abandonaram a empresa.

In [19]:
best_model = balanced_logistic_model

In [20]:
# Seleção do melhor modelo.
#
# Apesar da Logistic Regression normal apresentar maior Accuracy,
# a Logistic Regression balanceada foi escolhida porque identifica
# uma maior proporção de colaboradores que realmente abandonam a empresa.

## Comparação e Escolha do Modelo Final

Neste notebook foram testados três modelos de classificação para prever a variável `Attrition`, que indica se um colaborador abandonou ou permaneceu na empresa.

Os modelos testados foram:

1. Logistic Regression
2. Random Forest Classifier
3. Logistic Regression com `class_weight="balanced"`

A Logistic Regression normal foi utilizada como modelo baseline. Este modelo apresentou uma Accuracy elevada, cerca de 0.87, e um F1-Score de aproximadamente 0.52 para a classe `Yes`. Apesar do bom desempenho geral, o modelo teve dificuldade em identificar todos os colaboradores que realmente abandonaram a empresa.

De seguida foi testado o modelo Random Forest. Apesar de ser um modelo mais robusto em muitos problemas, neste caso apresentou pior desempenho na classe `Yes`, com Recall e F1-Score inferiores aos da Logistic Regression. Isto indica que o modelo favoreceu demasiado a classe maioritária `No`.

Por fim, foi testada a Logistic Regression com `class_weight="balanced"`. Esta abordagem foi utilizada porque o dataset está desequilibrado, existindo muitos mais casos `No` do que `Yes`.

Com o balanceamento, a Accuracy global diminuiu, mas o Recall da classe `Yes` aumentou significativamente. Isto significa que o modelo passou a identificar mais colaboradores que realmente abandonaram a empresa.

Como o objetivo principal do problema é prever o abandono de colaboradores, o modelo escolhido foi a **Logistic Regression com class_weight="balanced"**.

Apesar de apresentar menor Accuracy, este modelo é mais adequado ao objetivo do projeto, pois dá maior importância à identificação dos colaboradores com risco de abandono.

In [21]:
import pickle

with open("../models/G16_pipeline_classification.pkl", "wb") as file:
    pickle.dump(best_model, file)

In [22]:
# Exportação do modelo final de classificação.
#
# O modelo escolhido foi a Logistic Regression com class_weight="balanced",
# pois apresentou melhor capacidade de identificar colaboradores que
# abandonam a empresa.
#
# A pipeline completa é guardada num ficheiro .pkl, incluindo tanto o
# pré-processamento como o modelo treinado.
#
# Isto permite que o modelo seja reutilizado posteriormente para fazer
# previsões sobre novos dados sem repetir manualmente as transformações.

In [23]:
with open("../models/G16_pipeline_classification.pkl", "rb") as file:
    loaded_model = pickle.load(file)

test_predictions = loaded_model.predict(X_test)

print(test_predictions[:10])

['No' 'Yes' 'No' 'No' 'No' 'No' 'Yes' 'Yes' 'No' 'No']


In [24]:
# Validação do ficheiro exportado.
#
# O modelo guardado em .pkl foi novamente carregado e utilizado
# para gerar previsões no conjunto de teste.
#
# Esta etapa confirma que o ficheiro exportado está funcional.